# 01 — Data Understanding

**Project:** Credit Risk Model — Predicting Consumer Loan Default
**Dataset:** Give Me Some Credit (150,000 borrowers)

Goal of this notebook: load the data, understand its structure, quality, and
quirks *before* any modeling. Getting this right prevents garbage-in-garbage-out.

We'll cover:
1. Load & shape
2. Data types
3. Target variable (default rate) — note the imbalance
4. Missing values
5. A first look at distributions and suspicious values


In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_STATE = 42
DATA_PATH = "../data/raw/cs-training.csv"


## 1. Load the data

In [2]:
df = pd.read_csv(DATA_PATH, index_col=0)
print(f"Rows: {df.shape[0]:,}   Columns: {df.shape[1]}")
df.head()

Rows: 150,000   Columns: 11


,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
1,1,0.7661,45,2,0.8030,"9,120.0000",13,0,6,0,2.0000
2,0,0.9572,40,0,0.1219,"2,600.0000",4,0,0,0,1.0000
3,0,0.6582,38,1,0.0851,"3,042.0000",2,1,0,0,0.0000
4,0,0.2338,30,0,0.0360,"3,300.0000",5,0,0,0,0.0000
5,0,0.9072,49,1,0.0249,"63,588.0000",7,0,1,0,0.0000


## 2. Data types & structure

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 1 to 150000
Data columns (total 11 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   SeriousDlqin2yrs                      150000 non-null  int64  
 1   RevolvingUtilizationOfUnsecuredLines  150000 non-null  float64
 2   age                                   150000 non-null  int64  
 3   NumberOfTime30-59DaysPastDueNotWorse  150000 non-null  int64  
 4   DebtRatio                             150000 non-null  float64
 5   MonthlyIncome                         120269 non-null  float64
 6   NumberOfOpenCreditLinesAndLoans       150000 non-null  int64  
 7   NumberOfTimes90DaysLate               150000 non-null  int64  
 8   NumberRealEstateLoansOrLines          150000 non-null  int64  
 9   NumberOfTime60-89DaysPastDueNotWorse  150000 non-null  int64  
 10  NumberOfDependents                    146076 non-null  float64
dtypes: float64(

Note: everything is numeric. In the original competition, `MonthlyIncome` and
`NumberOfDependents` are the only columns with missing values.

In [4]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
SeriousDlqin2yrs,"150,000.0000",0.0668,0.2497,0.0000,0.0000,0.0000,0.0000,1.0000
RevolvingUtilizationOfUnsecuredLines,"150,000.0000",6.0484,249.7554,0.0000,0.0299,0.1542,0.5590,"50,708.0000"
age,"150,000.0000",52.2952,14.7719,0.0000,41.0000,52.0000,63.0000,109.0000
NumberOfTime30-59DaysPastDueNotWorse,"150,000.0000",0.4210,4.1928,0.0000,0.0000,0.0000,0.0000,98.0000
DebtRatio,"150,000.0000",353.0051,"2,037.8185",0.0000,0.1751,0.3665,0.8683,"329,664.0000"
MonthlyIncome,"120,269.0000","6,670.2212","14,384.6742",0.0000,"3,400.0000","5,400.0000","8,249.0000","3,008,750.0000"
NumberOfOpenCreditLinesAndLoans,"150,000.0000",8.4528,5.1460,0.0000,5.0000,8.0000,11.0000,58.0000
NumberOfTimes90DaysLate,"150,000.0000",0.2660,4.1693,0.0000,0.0000,0.0000,0.0000,98.0000
NumberRealEstateLoansOrLines,"150,000.0000",1.0182,1.1298,0.0000,0.0000,1.0000,2.0000,54.0000
NumberOfTime60-89DaysPastDueNotWorse,"150,000.0000",0.2404,4.1552,0.0000,0.0000,0.0000,0.0000,98.0000


## 3. The target: `SeriousDlqin2yrs`

1 = the borrower became 90+ days delinquent within 2 years.
This is an **imbalanced** classification problem — most borrowers do *not* default.


In [5]:
target = "SeriousDlqin2yrs"

print("Default rate: {:.2%}".format(df[target].mean()))
print()
print(df[target].value_counts().rename({0: "No default (0)", 1: "Default (1)"}).to_string())

Default rate: 6.68%

SeriousDlqin2yrs
No default (0)    139974
Default (1)        10026


**Why this matters:** with a ~7% positive rate, accuracy is a useless metric
(predicting "no default" for everyone is 93% accurate and worthless). We will lean on
**AUC, the KS statistic, and precision/recall** later, and handle imbalance with class weights.

## 4. Missing values

In [6]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"missing": missing, "pct": missing_pct}).query("missing > 0")

,missing,pct
MonthlyIncome,29731,19.8200
NumberOfDependents,3924,2.6200


`MonthlyIncome` is ~20% missing — that's substantial and will need a deliberate strategy.
`NumberOfDependents` is ~2.6% missing.

We'll defer imputation to notebook 03, but we must remember it here.

## 5. Early red flags

Credit datasets always contain sentinel/placeholder values. Two famous ones here:
- **Age of 0** (impossible)
- **Past-due counts of 96 or 98** (these are codes, not real counts)


In [7]:
print("Age == 0:", (df["age"] == 0).sum())
print("Age min  :", df["age"].min(), " max:", df["age"].max())
print()
for c in ["NumberOfTime30-59DaysPastDueNotWorse", "NumberOfTime60-89DaysPastDueNotWorse", "NumberOfTimes90DaysLate"]:
    vals = sorted(df[c].unique())
    print(f"{c}: max={df[c].max()}, has 96/98 = {any(v in (96, 98) for v in vals)}")

Age == 0: 1
Age min  : 0  max: 109

NumberOfTime30-59DaysPastDueNotWorse: max=98, has 96/98 = True
NumberOfTime60-89DaysPastDueNotWorse: max=98, has 96/98 = True
NumberOfTimes90DaysLate: max=98, has 96/98 = True


These anomalies (age 0, 96/98 codes) will be cleaned in notebook 03.

### Takeaway
- 150k borrowers, 10 features + target, all numeric.
- **Imbalanced target (~6.7%)** — drives our choice of metrics and loss.
- **Missing data** concentrated in `MonthlyIncome`.
- **Sentinel values** present — must be handled explicitly.

Next: **02 — Exploratory Data Analysis** to see which features actually separate
defaulters from non-defaulters.